In [1]:
import cv2
import face_recognition
import os
import serial
import time

try:
    # 🌟 請確認 'COM3' 是你在裝置管理員查到的正確號碼
    ser = serial.Serial('COM3', 115200, timeout=1)
    print("✅ 已成功連線至 STM32 控制板")
except Exception as e:
    print(f"⚠️ 未偵測到序列埠，系統將進入「無硬體模擬模式」. 錯誤: {e}")
    ser = None

# iPhone DroidCam 網址 (請確保手機和電腦連同一個 Wi-Fi)
video_url = "http://192.168.0.96:4747/video"
print("📱 準備連線至 iPhone 鏡頭...")

⚠️ 未偵測到序列埠，系統將進入「無硬體模擬模式」. 錯誤: could not open port 'COM3': FileNotFoundError(2, '系統找不到指定的檔案。', None, 2)
📱 準備連線至 iPhone 鏡頭...


In [2]:
known_face_encodings = []
known_face_names = []
dataset_path = "dataset"

print("⏳ 正在建立特徵資料庫...")
for person_name in os.listdir(dataset_path):
    person_folder = os.path.join(dataset_path, person_name)
    if not os.path.isdir(person_folder): continue

    for image_file in os.listdir(person_folder):
        if image_file.lower().endswith(('.jpg', '.jpeg', '.png')):
            image_path = os.path.join(person_folder, image_file)
            raw_img = face_recognition.load_image_file(image_path)

            # 🌟 修正版：寫出明確的常數名稱，避免 OpenCV 找不到屬性
            for angle in [0, 90, 180, 270]:
                if angle == 0:
                    test_img = raw_img
                elif angle == 90:
                    test_img = cv2.rotate(raw_img, cv2.ROTATE_90_CLOCKWISE)
                elif angle == 180:
                    test_img = cv2.rotate(raw_img, cv2.ROTATE_180)
                else: # 270
                    test_img = cv2.rotate(raw_img, cv2.ROTATE_90_COUNTERCLOCKWISE)

                encodings = face_recognition.face_encodings(test_img)
                if len(encodings) > 0:
                    known_face_encodings.append(encodings[0])
                    known_face_names.append(person_name)
                    break

print(f"🚀 建檔完成！共載入 {len(known_face_names)} 個特徵。")

⏳ 正在建立特徵資料庫...
🚀 建檔完成！共載入 10 個特徵。


In [3]:
import time

last_unlock_time = 0  # 紀錄上次開鎖時間，避免當機

def trigger_unlock(name):
    global last_unlock_time
    current_time = time.time()

    # 權限檢查：確認是 Joe，且距離上次開鎖超過 5 秒
    if name == "Joe" and ser and (current_time - last_unlock_time > 5):
        try:
            ser.write(b'1')
            print(f"🔓 歡迎 {name}！已發送開鎖指令給 STM32。")
            last_unlock_time = current_time
        except Exception as e:
            print(f"⚠️ 序列埠傳送失敗，請檢查連線: {e}")

print("✅ 開鎖控制模組 (Cell 3) 已載入準備完畢！")

✅ 開鎖控制模組 (Cell 3) 已載入準備完畢！


In [4]:
video_capture = cv2.VideoCapture(video_url)
print("🎥 系統啟動，開始監測鏡頭...")

while True:
    ret, frame = video_capture.read()
    if not ret:
        print("❌ 無法獲取影像，請檢查手機 DroidCam 連線或是 IP 網址是否正確。")
        break

    # 縮小畫面與轉換顏色以加速運算
    small_frame = cv2.resize(frame, (0, 0), fx=0.5, fy=0.5)
    rgb_small_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)

    # 尋找畫面中的人臉特徵
    face_locations = face_recognition.face_locations(rgb_small_frame)
    face_encodings = face_recognition.face_encodings(rgb_small_frame, face_locations)

    for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
        name = "Unknown"

        # 如果資料庫裡有資料，才進行比對
        if len(known_face_encodings) > 0:
            matches = face_recognition.compare_faces(known_face_encodings, face_encoding, tolerance=0.45)
            if True in matches:
                name = known_face_names[matches.index(True)]

                # 🌟 觸發開鎖指令 (裡面已包含 3 人權限與 5 秒防洗頻機制)
                trigger_unlock(name)

        # 繪製辨識框與名字 (座標還原乘 2)
        color = (0, 255, 0) if name != "Unknown" else (0, 0, 255)
        cv2.rectangle(frame, (left*2, top*2), (right*2, bottom*2), color, 2)
        cv2.putText(frame, name, (left*2+6, bottom*2-6), cv2.FONT_HERSHEY_DUPLEX, 0.8, (255, 255, 255), 1)

    # 顯示視窗
    cv2.imshow('Smart Locker System', frame)

    # 監聽按鍵，按下 'q' 結束程式
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# 釋放資源
video_capture.release()
cv2.destroyAllWindows()
print("🛑 系統已關閉。")

🎥 系統啟動，開始監測鏡頭...
❌ 無法獲取影像，請檢查手機 DroidCam 連線或是 IP 網址是否正確。
🛑 系統已關閉。


/* STM32F411 接收邏輯 (HAL 函式庫) */
// 放在 main.c 的 while(1) 迴圈中
uint8_t rx_data;
if (HAL_UART_Receive(&huart1, &rx_data, 1, 10) == HAL_OK) {
    if (rx_data == '1') {
        HAL_GPIO_WritePin(GPIOA, GPIO_PIN_5, GPIO_PIN_SET);   // 開鎖
        HAL_Delay(3000);                                     // 持續 3 秒
        HAL_GPIO_WritePin(GPIOA, GPIO_PIN_5, GPIO_PIN_RESET); // 關鎖
    }
}

video_capture = cv2.VideoCapture(video_url)

while True:
    ret, frame = video_capture.read()
    if not ret: break

    # 縮小畫面與轉換顏色
    small_frame = cv2.resize(frame, (0, 0), fx=0.5, fy=0.5)
    rgb_small_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)

    face_locations = face_recognition.face_locations(rgb_small_frame)
    face_encodings = face_recognition.face_encodings(rgb_small_frame, face_locations)

    for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
        name = "Unknown"
        matches = face_recognition.compare_faces(known_face_encodings, face_encoding, tolerance=0.45)

        if True in matches:
            name = known_face_names[matches.index(True)]
            # 🔓 辨識成功，傳送指令給 STM32
            if name == "Joe" and ser:
                ser.write(b'1')
                print("🔓 偵測到 Joe，發送開鎖指令！")

        # 繪製 UI (略，同之前版本)
        color = (0, 255, 0) if name != "Unknown" else (0, 0, 255)
        cv2.rectangle(frame, (left*2, top*2), (right*2, bottom*2), color, 2)

    cv2.imshow('Smart Locker System', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

video_capture.release()
cv2.destroyAllWindows()